# **Scraping Planos Infornet**

_Página do provedor: https://www.infornetfibra.com/planos-residencias_

In [11]:
# Bibliotecas
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date

## **1. Extração**

In [13]:
# Obter lista de planos
texto_html = open("../../data/scraped/infornet.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
container_planos = soup.find("div", class_="grid-cols-4")
planos = container_planos.find_all("div", class_="h-fit")
print(f"{len(planos)} cartões de planos capturados.")

10 cartões de planos capturados.


In [14]:
# Extrair informações dos planos
def raspar_plano(plano):
    velocidade = (plano.find("h1") or {}).get_text(strip=True)
    preco = (plano.find("strong") or {}).get_text(strip=True)
    svas = " ou ".join(
        t for t in (img.get("title") for img in plano.find_all("img"))
        if t
    ) or None

    return {
        "velocidade": velocidade or None,
        "preco": preco or None,
        "svas": svas
    }

df = pd.DataFrame(raspar_plano(p) for p in planos).drop_duplicates().reset_index(drop=True)
df

,velocidade,preco,svas
0,300,"89,90",NaN
1,500,"99,90",NaN
2,700,"129,90",Disney+ ou Globo Play ou Telecine
3,800,"149,90",Disney+ ou Globo Play ou Telecine ou Premiere
4,800,"159,90",Premiere
5,850,"159,90",NaN
6,900,"189,90",Telecine ou Globo Play
7,900,"179,90",Premiere
8,950,"199,90",NaN
9,1.1,"249,90",NaN


## **2. Tratamento**

In [15]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Infornet"

# Velocidade
df_tratado["VELOCIDADE"] = pd.to_numeric(df_tratado["velocidade"], errors="coerce")
df_tratado["VELOCIDADE"] = df_tratado["VELOCIDADE"].apply(
    lambda x: x * 1000 if pd.notna(x) and x < 10 else x
)
df_tratado["VELOCIDADE"] = df_tratado["VELOCIDADE"].round().astype("Int64")

# SVAs
df_tratado["SVAs"] = df_tratado["svas"]

# Preço: Substui vírgula por ponto
# Convert string em float
df_tratado["PRECO"] = (
    df_tratado["preco"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

,DATA,ISP,VELOCIDADE,SVAs,PRECO
0,2026-05-12,Infornet,300,NaN,89.9
1,2026-05-12,Infornet,500,NaN,99.9
2,2026-05-12,Infornet,700,Disney+ ou Globo Play ou Telecine,129.9
3,2026-05-12,Infornet,800,Disney+ ou Globo Play ou Telecine ou Premiere,149.9
4,2026-05-12,Infornet,800,Premiere,159.9
5,2026-05-12,Infornet,850,NaN,159.9
6,2026-05-12,Infornet,900,Telecine ou Globo Play,189.9
7,2026-05-12,Infornet,900,Premiere,179.9
8,2026-05-12,Infornet,950,NaN,199.9
9,2026-05-12,Infornet,1100,NaN,249.9


## **3. Carga**

In [16]:
# Salvando
provedor = "INFORNET.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")

🟢 2026-05-12-INFORNET.csv salvo!
